In [57]:
import pandas as pd

df = pd.read_csv('/Users/jeremy/Documents/protests_chai/frames/truncated_samples.csv.gz')

df.columns

Index(['row_id', 'text-topic', 'text-topic-exp', 'text-entity-name',
       'text-entity-gender', 'text-entity-sentiment',
       'text-entity-sentiment-exp', 'text-generic-frame',
       'text-generic-frame-exp', 'text-issue-frame', 'text-issue-frame-exp',
       'uuid', 'title', 'img-generic-frame', 'img-frame-exp',
       'img-entity-name', 'img-entity-gender', 'img-entity-sentiment',
       'img-entity-sentiment-exp', 'image-url', 'gpt-topic', 'authors',
       'date_publish', 'description', 'language', 'maintext', 'source_domain',
       'url', 'political_leaning', 'month', 'keyword_matches',
       'keyword_match_count', 'word_count', 'label', 'protest_excerpt',
       'excerpt_length'],
      dtype='str')

In [58]:
sample1 = df.sample(n=40, random_state=42)

remaining = df.drop(sample1.index)
sample2 = remaining.sample(n=40, random_state=45)
sample2 = sample2[['month', 'uuid', 'title', 'protest_excerpt', 'keyword_matches']]

sample2.to_csv('/Users/jeremy/Documents/protests_chai/frames/sample_1.csv', index=False)

remaining = remaining.drop(sample2.index)
sample3 = remaining.sample(n=40, random_state = 46)
sample3 = sample3[['month', 'uuid', 'title', 'protest_excerpt', 'keyword_matches']]
sample3.to_csv('/Users/jeremy/Documents/protests_chai/frames/sample_3.csv', index=False)

remaining = remaining.drop(sample3.index)
sample4 = remaining.sample(n=60, random_state = 47)
sample4 = sample4[['month', 'uuid', 'title', 'protest_excerpt', 'keyword_matches']]
sample4.to_csv('/Users/jeremy/Documents/protests_chai/frames/sample_4.csv', index=False)

In [47]:
sample3.head()

,month,uuid,title,protest_excerpt,keyword_matches
1172,2023-08-01_2023-08-31,f983cdea-747e-4387-982f-3d4f794f93db,US publisher of pro-fascist books revealed as ...,The listing also associated Ross with a Proton...,"['Riot', 'rally', 'demonstrators', 'rally']"
375,2023-08-01_2023-08-31,46959c59-3873-4534-9011-cae228ee02ce,7 prominent advocates charged in Hong Kong’s 2...,The Court of Appeals has overturned certain co...,"['protests', 'protests', 'PROTESTS', 'protests..."
4537,2023-07-01_2023-07-31,40de878d-a43f-4af9-bd06-3a5e18e73095,Tens of thousands of Israelis march as vote on...,Tens of thousands of Israelis opposed to a jud...,"['protesters', 'protests', 'Protesters', 'rally']"
1331,2023-09-01_2023-09-30,34fdaa3c-3834-4201-a4b0-a5e4b2d41cbe,Canada killing adds to suspicions of Indian cr...,Pakistan has previously accused India of carry...,"['protests', 'protesters', 'protesters']"
3384,2024-04-01_2024-04-30,24f1401c-81c5-4b4d-871d-ceaa0c88352d,Hezbollah terror flag found at Princeton's ant...,Students at elite college campuses across the ...,"['marches', 'protests', 'protesters', 'protest..."


In [42]:
import re
import pandas as pd
import simpledorff


def parse_excel_column(text):
    """
    Paste a single Excel column (numbers or text labels).
    Works with line breaks or tabs copied directly from Excel.
    """
    return [
        x.strip()
        for x in text.strip().splitlines()
        if x.strip()
    ]

def calculate_agreement(coder_a_text, coder_b_text, label_name="label"):
    coder_a = parse_excel_column(coder_a_text)
    coder_b = parse_excel_column(coder_b_text)

    if len(coder_a) != len(coder_b):
        raise ValueError(
            f"{label_name}: unequal lengths "
            f"({len(coder_a)} vs. {len(coder_b)})"
        )

    if len(coder_a) == 0:
        raise ValueError(f"{label_name}: no values were found.")

    wide = pd.DataFrame({
        "unit": range(len(coder_a)),
        "coder_a": coder_a,
        "coder_b": coder_b
    })

    raw_agreement = (wide["coder_a"] == wide["coder_b"]).mean()

    long = wide.melt(
        id_vars="unit",
        value_vars=["coder_a", "coder_b"],
        var_name="annotator",
        value_name="label"
    )

    alpha = simpledorff.calculate_krippendorffs_alpha_for_df(
        long,
        experiment_col="unit",
        annotator_col="annotator",
        class_col="label"
    )

    return {
        "field": label_name,
        "n": len(wide),
        "raw_agreement": raw_agreement,
        "krippendorff_alpha": float(alpha)
    }

In [53]:
jeremy_title_frame = """
no
yes
yes
yes
yes
yes
no
yes
no
yes
no
no
yes
no
yes
no
no
no
yes
no
no
no
yes
no
no
no
no
no
yes
yes
no
no
yes
no
yes
no
yes
yes
no
no
"""

julia_title_frame = """
no
yes
yes
yes
yes
yes
no
yes
no
yes
no
no
yes
no
no
no
no
no
yes
no
no
no
yes
no
no
no
no
no
yes
yes
no
no
yes
no
yes
no
yes
yes
no
yes
"""

result = calculate_agreement(
    jeremy_title_frame,
    julia_title_frame,
    label_name="thematic_present"
)

print(result)

with open("agreement_results.txt", "a") as f:
    f.write(
        f"{result['field']}: "
        f"n={result['n']}, "
        f"Raw Agreement={result['raw_agreement']:.3f}, "
        f"Krippendorff Alpha={result['krippendorff_alpha']:.3f}\n"
    )

{'field': 'thematic_present', 'n': 40, 'raw_agreement': np.float64(0.95), 'krippendorff_alpha': 0.8989769820971867}


In [55]:
import pandas as pd
import simpledorff

# ---------------------------------------------------------
# 1. Load Excel file
# ---------------------------------------------------------

df = pd.read_excel("pooled_annotations.xlsx", sheet_name=0)

tasks = [
    "title_frame",
    "excerpt_frame",
    "demands_present",
    "protester_quote",
    "authority_quote",
    "thematic_present"
]

# ---------------------------------------------------------
# 2. Basic checks
# ---------------------------------------------------------

required_columns = ["item_id", "annotator_id"] + tasks

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"These required columns are missing: {missing_columns}"
    )

# ---------------------------------------------------------
# 3. Normalize text without converting missing cells to "nan"
# ---------------------------------------------------------

def normalize_text(value):
    """
    Standardize text labels while preserving genuinely missing values.
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    # Optional: collapse repeated internal spaces
    value = " ".join(value.split())

    if value in {"", "nan", "none", "n/a", "na"}:
        return pd.NA

    return value


for task in tasks:
    df[task] = df[task].map(normalize_text)

df["annotator_id"] = df["annotator_id"].astype(str).str.strip()
df["item_id"] = df["item_id"].astype(str).str.strip()

# ---------------------------------------------------------
# 4. Convert from wide format to long format
# ---------------------------------------------------------

long_df = df.melt(
    id_vars=["item_id", "annotator_id"],
    value_vars=tasks,
    var_name="task",
    value_name="value"
)

# Remove missing annotations
long_df = long_df.dropna(subset=["value"]).copy()

# Each item-task combination is one reliability unit
long_df["unit_id"] = (
    long_df["item_id"]
    + "::"
    + long_df["task"]
)

# ---------------------------------------------------------
# 5. Create task-specific pooled labels
# ---------------------------------------------------------
# This prevents labels from different questions from being treated
# as the same category merely because they use the same text.
#
# Example:
# title_frame::violence
# demands_present::yes

long_df["pooled_value"] = (
    long_df["task"]
    + "::"
    + long_df["value"]
)

# ---------------------------------------------------------
# 6. Verify that each completed unit has two annotators
# ---------------------------------------------------------

counts = long_df.groupby("unit_id")["annotator_id"].nunique()

incomplete_units = counts[counts != 2]

if not incomplete_units.empty:
    print("Warning: these units do not have exactly two annotations:")
    print(incomplete_units)

    # Keep only units coded by exactly two distinct annotators
    valid_units = counts[counts == 2].index
    pooled_df = long_df[
        long_df["unit_id"].isin(valid_units)
    ].copy()
else:
    pooled_df = long_df.copy()

# ---------------------------------------------------------
# 7. Pooled raw agreement
# ---------------------------------------------------------

agreement_by_unit = pooled_df.groupby("unit_id")["pooled_value"].agg(
    lambda values: values.nunique() == 1
)

pooled_raw_agreement = agreement_by_unit.mean()

# ---------------------------------------------------------
# 8. Pooled Krippendorff's alpha
# ---------------------------------------------------------

pooled_alpha = simpledorff.calculate_krippendorffs_alpha_for_df(
    pooled_df,
    experiment_col="unit_id",
    annotator_col="annotator_id",
    class_col="pooled_value"
)

print(f"Pooled raw agreement: {pooled_raw_agreement:.3f}")
print(f"Pooled Krippendorff's alpha: {pooled_alpha:.3f}")

# ---------------------------------------------------------
# 9. Per-task raw agreement and alpha
# ---------------------------------------------------------

results = []

for task in tasks:
    sub = long_df[long_df["task"] == task].copy()

    task_counts = sub.groupby("unit_id")["annotator_id"].nunique()
    valid_task_units = task_counts[task_counts == 2].index

    sub = sub[sub["unit_id"].isin(valid_task_units)]

    if sub.empty:
        print(f"Skipping {task}: no complete annotation pairs.")
        continue

    raw_agreement = (
        sub.groupby("unit_id")["value"]
        .agg(lambda values: values.nunique() == 1)
        .mean()
    )

    alpha = simpledorff.calculate_krippendorffs_alpha_for_df(
        sub,
        experiment_col="unit_id",
        annotator_col="annotator_id",
        class_col="value"
    )

    results.append({
        "task": task,
        "n_units": sub["unit_id"].nunique(),
        "raw_agreement": raw_agreement,
        "krippendorff_alpha": alpha
    })

results_df = pd.DataFrame(results)

print("\nPer-task agreement:")
print(results_df.round(3).to_string(index=False))

Pooled raw agreement: 0.921
Pooled Krippendorff's alpha: 0.915

Per-task agreement:
            task  n_units  raw_agreement  krippendorff_alpha
     title_frame       40          0.925               0.875
   excerpt_frame       40          0.850               0.824
 demands_present       40          0.925               0.828
 protester_quote       40          0.925               0.840
 authority_quote       40          0.950               0.897
thematic_present       40          0.950               0.899
